In [1]:
from esm_msr.preprocessing import MegaScaleDatasetPreprocessor

preprocessor = MegaScaleDatasetPreprocessor('/home/sareeves/software/esm-msr/data/tsuboyama/Tsuboyama2023_Dataset2_Dataset3_20230416.csv', '/home/sareeves/software/esm-msr/data/tsuboyama/AlphaFold_model_PDBs', spurs_override=True)
#split_name = 'march11_megascale'
split_name = 'mega_splits'
splits = preprocessor.create_training_splits(f'/home/sareeves/software/esm-msr/data/{split_name}.pkl')
splits = preprocessor.get_splits()

/home/sareeves/miniconda3/envs/msr_venv/lib/python3.11/site-packages/Bio/pairwise2.py:278: BiopythonDeprecationWarning: Bio.pairwise2 has been deprecated, and we intend to remove it in a future release of Biopython. As an alternative, please consider using Bio.Align.PairwiseAligner as a replacement, and contact the Biopython developers if you still need the Bio.pairwise2 module.
  warnings.warn(
2026-04-28 13:34:24,918 - INFO - Preprocessing according to SPURS methodology.
2026-04-28 13:34:32,451 - INFO - Total size: 550141 according to SPURS preprocessing
2026-04-28 13:34:32,480 - INFO - Using split file /home/sareeves/software/esm-msr/data/mega_splits.pkl.
Creating train datasets:  13%|█▎        | 30/239 [00:08<00:56,  3.71it/s]2026-04-28 13:34:41,562 - INFO - removed 111 rows from the dataset due to SPURS filtering
2026-04-28 13:34:41,572 - WARNING - Removed 108 fake or improper mutations (wt == mut)
Creating train datasets:  14%|█▍        | 34/239 [00:09<00:54,  3.74it/s]2026-04-28

In [2]:
len(splits['train']), len(splits['val']), len(splits['test'])

(216, 29, 30)

In [3]:
def revert_mutation(row):
    try:
        wt = row['fr1']
        pos = int(row['pos1'])
        mut = row['to1']
    except:
        print(row)
    mutseq = row['mut_seq']
    assert mutseq[pos-1] == mut
    mutseq = list(mutseq)
    mutseq[pos-1] = wt
    row = row.dropna()
    try:
        wt = row['fr2']
        pos = int(row['pos2'])
        mut = row['to2']
        assert mutseq[pos-1] == mut
        mutseq = list(mutseq)
        mutseq[pos-1] = wt
    except KeyError:
        pass
    return ''.join(mutseq)

def revert_mut_bb(row):
    if type(row['mut_structure']) != str:
        return row['wt_seq']
    else:
        wt = row['mut_structure'][0]
        pos = int(row['mut_structure'][1:-1])
        mut = row['mut_structure'][-1]
        mutseq = row['wt_seq']
        try:
            assert mutseq[pos-1] == mut, row
            mutseq = list(mutseq)
            mutseq[pos-1] = wt
            row = row.dropna()
        except:
            assert row['code_wt'] == '1UBQ'
            mutseq = list('MQIFVKTLTGKTITSEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSDYNIQKESTLHLVLR')
        print('Reverted', row['code_wt'], row['mut_structure'], ''.join(mutseq))
        return ''.join(mutseq)

In [4]:
split_name += '_remove_spurs'

In [6]:
import os
data = preprocessor.split_dfs

os.makedirs(f'/home/sareeves/software/MutateEverything/data/{split_name}/', exist_ok=True)
with open(f'/home/sareeves/software/MutateEverything/data/{split_name}/{split_name}_seqs.fasta', 'w') as fa:
    with open(f'/home/sareeves/software/MutateEverything/data/{split_name}/{split_name}_wrong_seqs.fasta', 'w') as fb:
        for t in ['train', 'val', 'test']:

            df_cur = data[t]

            df_cur['pdb_id'] = df_cur['code_wt'] + df_cur['chain']

            df_cur['ddg'] = df_cur['ddG_ML']
            df_cur['mut_info'] = df_cur['mut_type']
            df_cur['mut_seq'] = df_cur['aa_seq']

            df_cur['fr1'] = df_cur['mut_info'].apply(lambda x: x.split(':')[0][0])
            df_cur['pos1'] = df_cur['mut_info'].apply(lambda x: x.split(':')[0][1:-1])
            df_cur['to1'] = df_cur['mut_info'].apply(lambda x: x.split(':')[0][-1])

            df_cur.loc[df_cur['mut_info'].str.contains(':'), 'fr2'] = df_cur.loc[df_cur['mut_info'].str.contains(':'), 'mut_info'].apply(lambda x: x.split(':')[1][0])
            df_cur.loc[df_cur['mut_info'].str.contains(':'), 'pos2'] = df_cur.loc[df_cur['mut_info'].str.contains(':'), 'mut_info'].apply(lambda x: x.split(':')[1][1:-1])
            df_cur.loc[df_cur['mut_info'].str.contains(':'), 'to2'] = df_cur.loc[df_cur['mut_info'].str.contains(':'), 'mut_info'].apply(lambda x: x.split(':')[1][-1])

            df_cur['wt_seq'] = df_cur.apply(lambda x: revert_mutation(x), axis=1)
            df_cur['wt_seq'] = df_cur.apply(lambda x: revert_mut_bb(x), axis=1)

            #df_cur[['pdb_id', 'ddg', 'mut_info', 'mut_seq', 'fr1', 'pos1', 'to1', 'fr2', 'pos2', 'to2', 'wt_seq']]

            os.makedirs(f'/home/sareeves/software/MutateEverything/data/{split_name}/mutations', exist_ok=True)
            df_cur.to_csv(f'/home/sareeves/software/MutateEverything/data/{split_name}/mutations/{split_name}_{t}.csv')

            for (code, chain), group in df_cur.groupby(['code_wt', 'chain']):
                os.makedirs(f'/home/sareeves/software/MutateEverything/data/{split_name}/fasta/{code}{chain}', exist_ok=True)
                with open(f'/home/sareeves/software/MutateEverything/data/{split_name}/fasta/{code}{chain}/{code}{chain}.fasta', 'w') as f:
                    wt_seq = group['wt_seq'].head(1).item()
                    f.write(f'>{code}{chain}\n{wt_seq}')
                    fa.write(f'>{code}{chain}\n{wt_seq}\n')
                    if code != group.reset_index()['code'].head(1).item():
                        print(code)
                        fb.write(f'>{code}{chain}\n{wt_seq}\n')

                os.makedirs(f'/home/sareeves/software/MutateEverything/data/{split_name}/msa/{code}{chain}', exist_ok=True)

Reverted 1OPS V44A SQSVVATQLIPMNTALTPAMMEGKVTNPIGIPFAEMSQLVGKQVNTPVAKGQTLMPNMVKTYA
Reverted 1OPS T53S SQSVVATQLIPMNTALTPAMMEGKVTNPIGIPFAEMSQLVGKQVNTPVAKGQTLMPNMVKTYA
Reverted 1OPS V44A SQSVVATQLIPMNTALTPAMMEGKVTNPIGIPFAEMSQLVGKQVNTPVAKGQTLMPNMVKTYA
Reverted 1OPS T53S SQSVVATQLIPMNTALTPAMMEGKVTNPIGIPFAEMSQLVGKQVNTPVAKGQTLMPNMVKTYA
Reverted 1OPS V44A SQSVVATQLIPMNTALTPAMMEGKVTNPIGIPFAEMSQLVGKQVNTPVAKGQTLMPNMVKTYA
Reverted 1OPS T53S SQSVVATQLIPMNTALTPAMMEGKVTNPIGIPFAEMSQLVGKQVNTPVAKGQTLMPNMVKTYA
Reverted 1OPS V44A SQSVVATQLIPMNTALTPAMMEGKVTNPIGIPFAEMSQLVGKQVNTPVAKGQTLMPNMVKTYA
Reverted 1OPS T53S SQSVVATQLIPMNTALTPAMMEGKVTNPIGIPFAEMSQLVGKQVNTPVAKGQTLMPNMVKTYA
Reverted 1OPS V44A SQSVVATQLIPMNTALTPAMMEGKVTNPIGIPFAEMSQLVGKQVNTPVAKGQTLMPNMVKTYA
Reverted 1OPS T53S SQSVVATQLIPMNTALTPAMMEGKVTNPIGIPFAEMSQLVGKQVNTPVAKGQTLMPNMVKTYA
Reverted 1OPS V44A SQSVVATQLIPMNTALTPAMMEGKVTNPIGIPFAEMSQLVGKQVNTPVAKGQTLMPNMVKTYA
Reverted 1OPS T53S SQSVVATQLIPMNTALTPAMMEGKVTNPIGIPFAEMSQLVGKQVNTPVAKGQTLMPNMVKTYA
Reve

In [ ]:
STOP

NameError: name 'STOP' is not defined

In [ ]:
df.columns

Index(['aa_seq', 'mut_type', 'WT_name', 'dG_ML', 'ddG_ML'], dtype='object')

In [ ]:
import pandas as pd

root_path = '/home/sareeves/software/SPURS/'

fname = os.path.join(root_path,'data/dataset/megascale/Tsuboyama2023_Dataset2_Dataset3_20230416.csv')
df = pd.read_csv(fname, usecols=["ddG_ML", "mut_type", "WT_name", "aa_seq", "dG_ML"])
# remove unreliable data and more complicated mutations
df = df.loc[df.ddG_ML != '-', :].reset_index(drop=True)
df = df.loc[~df.mut_type.str.contains("ins") & ~df.mut_type.str.contains("del") & ~df.mut_type.str.contains(":"), :].reset_index(drop=True)

mmseq_wt_search = os.path.join(root_path,'data/dataset/megascale/mmseq_mut_search_0.25.m8')
ret = set()
with open(mmseq_wt_search, 'r') as f:
    for line in f.readlines():
        first_column_value = int(line.split("\t")[0])
        second_column_value = int(line.split("\t")[1])
        ret.add(first_column_value)
        ret.add(second_column_value)
print(len(list(set(ret))))
ret = list(set(ret))
# we dont want the rows in the ret
previous_len = len(df)
orig_codes = list(df['WT_name'].unique())
df_removed = df.loc[df.index.isin(ret), :]
df = df.loc[~df.index.isin(ret), :].reset_index(drop=True)
new_len = len(df)
print(previous_len - new_len)
new_codes = list(df['WT_name'].unique())
removed_codes = set(orig_codes).difference(new_codes)
print(len(ret))
print(removed_codes)
print(set(df.loc[df.index.isin(ret), 'WT_name']))

30887
30887
30887
set()
{'1UCS.pdb_M59K', '1GJS.pdb', 'HHH_rd4_0395.pdb', 'GG|run6_0851_0003.pdb', '2KVS.pdb', '2KWH.pdb', '2L33.pdb', '2K52.pdb_L42A', '2KYB.pdb_G2S', '2KT8.pdb_G54S', '2KVT.pdb', '6EWU.pdb', 'GG|run4_1050_0002.pdb', '1QP2.pdb', '1PGX.pdb_G42A', '2JN4.pdb', '2KRU.pdb', '1OPS.pdb_T53S', '6IWS.pdb', '2K5H.pdb_G23A', 'HEEH_KT_rd6_0793.pdb', 'EEHEE_rd3_1818.pdb', '5KPH.pdb', 'GG|run3_1315_0003.pdb', '2LSS.pdb_G6S', 'GG|run4_0467_0002.pdb', '2L7M.pdb', '1SRM.pdb', 'HEEH_rd3_0055.pdb', '2MKX.pdb_N29S', '2L7F.pdb', '1PWT.pdb', '1URF.pdb', 'v2_2HDZ.pdb', '2JVG.pdb', '2MI6.pdb', '2LYP.pdb_P41A', '1SF0.pdb_V59K', 'GG|run5_1309_0004.pdb', '1S1N.pdb_V29A', '2M6Y.pdb', '2M7O.pdb_I21A', '2KRS.pdb_Y53S', 'GG|run5_1147_0004.pdb', '2CDT.pdb', 'GG|run3_1365_0003.pdb', '2MCH.pdb', 'GG|run6_1310_0004.pdb', '1SIF.pdb_F13S', '2MA4.pdb', '2LX2.pdb_I32A', '2MKY.pdb', 'EA|run5_0050_0004.pdb', '2B88.pdb_pross1', '2JTV.pdb', '2LYP.pdb_V26S', '2JZ2.pdb', 'EEHEE_rd4_0470.pdb', '2MKX.pdb_L11A', 'HH

In [ ]:
dgr = df_removed.groupby('WT_name').count().iloc[:, 2:3]

In [ ]:
dg = df.groupby('WT_name').count().iloc[:, 0:1]

In [ ]:
dgr.join(dg, how='left').rename(columns={'dG_ML': 'removed', 'aa_seq': 'remaining'})

,removed,remaining
WT_name,,
1A0N.pdb_L7S,328,769
1A0N.pdb_V55A,549,551
1BK2.pdb_F47A,19,880
1BK2.pdb_L5S,34,923
1CSQ.pdb_F49A,757,511
...,...,...
GG|run5_0637_0006.pdb,132,757
GG|run5_1147_0004.pdb,58,836
GG|run5_1309_0004.pdb,74,810


In [ ]:
df_removed

,aa_seq,mut_type,WT_name,dG_ML,ddG_ML
8,HEVTIHLGDKTIRVDGLDKELLEILKELARRGADEEELRKEIERWER,D1H,EA|run2_0325_0005.pdb,3.234823167049992,-0.08010508773722425
9,REVTIHLGDKTIRVDGLDKELLEILKELARRGADEEELRKEIERWER,D1R,EA|run2_0325_0005.pdb,2.912039319692404,-0.40288893509481216
10,KEVTIHLGDKTIRVDGLDKELLEILKELARRGADEEELRKEIERWER,D1K,EA|run2_0325_0005.pdb,3.120279807494329,-0.1946484472928871
11,TEVTIHLGDKTIRVDGLDKELLEILKELARRGADEEELRKEIERWER,D1T,EA|run2_0325_0005.pdb,3.369760506235392,0.05483225144817583
12,SEVTIHLGDKTIRVDGLDKELLEILKELARRGADEEELRKEIERWER,D1S,EA|run2_0325_0005.pdb,3.276337081360694,-0.038591173426522296
...,...,...,...,...,...
379628,NGDKGYNGLAEAKEKAIKDLKIYGIGEHYIKLIEKAKQVAAVEDLK...,A52I,2MH8.pdb,1.1452639009926182,0.35834774779048384
379629,NGDKGYNGLAEAKEKAIKDLKIYGIGEHYIKLIEKAKQVAAVEDLK...,A52W,2MH8.pdb,1.1771910667384458,0.3902749135363114
379630,NGDKGYNGLAEAKEKAIKDLKIYGIGEHYIKLIEKAKQVAAVEDLK...,A52Y,2MH8.pdb,0.9568695470265915,0.16995339382445718
379631,NGDKGYNGLAEAKEKAIKDLKIYGIGEHYIKLIEKAKQVAAVEDLK...,A52F,2MH8.pdb,0.9494958627466972,0.1625797095445628


In [ ]:
s = 'DVEPGKFYKGVVTRIEKYGAFINLNEQVRGLLRPRDMISLRLENLNVGDEIIVQAIDVRPEKREIDFKYIP'
s[42-1]

'L'

In [ ]:
s = 'DVEPGKFYKGVVTRIEKYGAFINLNEQVRGLLRPRDMISLRLENLNVGDEIIVQAIDVRPEKREIDFKYIP'
s[42-1]

'L'

In [ ]:
s = 'AVSDRLIGRKGVVMEAISPQNSGLVKVDGETWRATSGTVLDVGEEVSVKAIEGVKLVVEKLE'
s[11-1]

'G'

In [ ]:
import pandas as pd
import os
import shutil
from tqdm.notebook import tqdm

import re

def parse_mutation_column_to_separate_columns(df, column_name):
    """
    Parse a DataFrame column containing mutation strings into separate columns
    for each mutation component (fr1, pos1, to1, fr2, pos2, to2, etc.)
    and additional list columns.
    
    Args:
        df (pandas.DataFrame): DataFrame containing the mutation column
        column_name (str): Name of column containing mutation strings
    
    Returns:
        pandas.DataFrame: DataFrame with additional columns for parsed mutations
    """
    # Create a copy to avoid modifying the original
    result_df = df.copy()
    
    # First, determine the maximum number of mutations in any entry
    max_mutations = 0
    for mutation_string in df[column_name]:
        if pd.isna(mutation_string) or mutation_string == '':
            continue
        
        mutations = mutation_string.split(':')
        max_mutations = max(max_mutations, len(mutations))
    
    # Initialize empty columns for each mutation component
    for i in range(1, max_mutations + 1):
        result_df[f'fr{i}'] = None
        result_df[f'pos{i}'] = None
        result_df[f'to{i}'] = None
    
    # Initialize the list columns
    result_df['mut_fr_list'] = [[] for _ in range(len(df))]
    result_df['mut_pos_list'] = [[] for _ in range(len(df))]
    result_df['mut_to_list'] = [[] for _ in range(len(df))]
    
    # Parse each row
    for idx, mutation_string in enumerate(df[column_name]):
        if pd.isna(mutation_string) or mutation_string == '':
            continue
            
        mutations = mutation_string.split(':')
        
        fr_list = []
        pos_list = []
        to_list = []
        
        for i, part in enumerate(mutations, 1):
            # Extract components using regex
            match = re.match(r'([A-Za-z])(\d+)([A-Za-z])', part)
            
            if match:
                from_aa = match.group(1)
                position = match.group(2)  # Keep as string
                to_aa = match.group(3)
                
                # Update individual columns
                result_df.at[idx, f'fr{i}'] = from_aa
                result_df.at[idx, f'pos{i}'] = int(position)  # Still convert to int for individual columns
                result_df.at[idx, f'to{i}'] = to_aa
                
                # Add to lists for collective columns
                fr_list.append(from_aa)
                pos_list.append(position)  # Keep as string for the list
                to_list.append(to_aa)
            else:
                print(f"Warning: Could not parse mutation '{part}'")
        
        # Update list columns
        result_df.at[idx, 'mut_fr_list'] = fr_list
        result_df.at[idx, 'mut_pos_list'] = pos_list
        result_df.at[idx, 'mut_to_list'] = to_list
    
    return result_df

In [ ]:
import pandas as pd
import os
import shutil
from tqdm.notebook import tqdm

with open(f'/home/sareeves/software/MutateEverything/data/domainome/domainome_seqs.fasta', 'w') as fa:

    for df_path in [#'../data/preprocessed/s669_mapped.csv',
                    #'../data/preprocessed/q3421_mapped.csv',
                    #'../data/preprocessed/k3822_mapped.csv',
                    '/home/sareeves/PSLMs/data/domainome1/domainome_mapped_2026.csv'
                    #'../data/tsuboyama/tsuboyama_all_subs_corrected.csv']:
                    #'../data/preprocessed/ssym_mapped.csv',
                    #'../data/preprocessed/ptmul_mapped.csv',
                    #'../data/preprocessed/fireprot_mapped_new.csv',
                    #'../data/preprocessed/s783_mapped.csv',
                    #'../data/preprocessed/s571_mapped.csv',
                    #'../data/preprocessed/s2648_mapped.csv',
                    #'../data/preprocessed/s8754_mapped.csv',
                    ]:
        
        df = pd.read_csv(df_path, index_col=0)

        if 'domainome' in df_path:
            df['mut_seq'] = df['aa_seq']
            df = df.dropna(subset='wt_seq') 
            df['ddg'] = df['scaled_fitness']
            df['code'] = df['domain_ID']
            df['chain'] = 'A'
            df['seq_pos'] = df['position']
            #df['wt_seq_full'] = df['uniprot_seq']
            #df['mut_seq_full'] = df.apply(lambda x: apply_mutation_uniprot(x), axis=1) 
            #df['position'] -= df['offset_up']

        else:
            #df['mut_seq'] = df['pdb_ungapped']
            #df['wt_seq_'] = df.apply(lambda x: revert_mutation(x), axis=1)
            try:
                df['ddg'] = df['ddG']
            except:
                df['ddg'] = df['dTm']

            #for (wt_seq_, uniprot_seq), group in tqdm(df.groupby(['wt_seq_', 'uniprot_seq'])):
            #    wt_seq, aligned_positions, gap_positions = reduce_seq(group.head(1))
            #    df.loc[(df['wt_seq_']==wt_seq_) & (df['uniprot_seq']==uniprot_seq), 'wt_seq'] = wt_seq
            #    df.loc[(df['wt_seq_']==wt_seq_) & (df['uniprot_seq']==uniprot_seq), 'aligned_positions'] = str(aligned_positions)
            #    df.loc[(df['wt_seq_']==wt_seq_) & (df['uniprot_seq']==uniprot_seq), 'gap_positions'] = str(gap_positions)
            #    #pd.DataFrame(group.apply(lambda x: reduce_seq2(x), axis=1).tolist(), index=df.index)
            #if 'ssym' in df_path:
            #    df['wt_seq'] = df['wt_seq_']
            #df.loc[df['code']=='2N53', 'wt_seq'] = df.loc[df['code']=='2N53', 'wt_seq_']
            #df.loc[df['code']=='2SPZ', 'wt_seq'] = df.loc[df['code']=='2N53', 'wt_seq_']

            #print(df['wt_seq'])
            #assert all(df['wt_seq_'].str.len()==df['wt_seq'].str.len())

        df['pdb_id'] = df['code'] + df['chain']
        try:
            df['mut_info'] = df['wild_type'] + df['seq_pos'].astype(int).astype(str) + df['mutation']
        except:
            assert 'mut_info' in df.columns
            df['mut_info'] = df['mut_info_seq_pos']
        #df['pos'] = df['position'] #if 'domainome' not in name else df['position'] - df['offset_up']
        #df.to_csv(df_path.replace('mapped.csv', 'mapped_new.csv'))
        
        name = df_path.split('/')[-1].split('_')[0]
        os.makedirs(f'/home/sareeves/software/MutateEverything/data/domainome/mutations/', exist_ok=True)
        if 'domainome' not in df_path:
            df[['wt_seq', 'mut_seq', 'ddg', 'pdb_id', 'mut_info']].to_csv(f'/home/sareeves/software/MutateEverything/data/domainome/mutations/{name}.csv')
        else:
            df[['wt_seq', 'mut_seq', 'ddg', 'pdb_id', 'mut_info', 'domain_ID', 'uniprot_ID']].to_csv(f'/home/sareeves/software/MutateEverything/data/{name}/mutations/{name}.csv')

        if 'ptmul' in df_path:
            df2 = pd.read_csv(f'/home/sareeves/software/MutateEverything/data/domainome/mutations/{name}.csv')
            df2 = parse_mutation_column_to_separate_columns(df2, 'mut_info').set_index('uid')
            df2.to_csv(f'/home/sareeves/software/MutateEverything/data/domainome/mutations/{name}.csv')

        for (code, chain), group in df.groupby(['code', 'chain']):
            os.makedirs(f'/home/sareeves/software/MutateEverything/data/domainome/fasta/{code}{chain}', exist_ok=True)
            with open(f'/home/sareeves/software/MutateEverything/data/domainome/fasta/{code}{chain}/{code}{chain}.fasta', 'w') as f:
                # IMPORTANT: can reassess using wt_seq_, which is the reverted pdb seq. will probably make some difference
                #if 'ssym' in df_path or code == '2N53' or code == '2SPZ':
                #    # reverted pdb seq
                #    wt_seq = group['wt_seq_'].head(1).item()
                #else:
                    # sliced uniprot seq (forms MSA)
                wt_seq = group['wt_seq'].head(1).item()
                f.write(f'>{code}{chain}\n{wt_seq}')
                fa.write(f'>{code}{chain}\n{wt_seq}\n')
        # os.makedirs(f'/home/sareeves/software/MutateEverything/data/{name}/msa/{code}{chain}', exist_ok=True)
        # process_msa_with_biopython(group['reduced_msa_file'].head(1).item(), 
        #                            group['wt_seq'].head(1).item(), 
        #                            group['aligned_positions'].head(1).item(),
        #                            group['gap_positions'].head(1).item(),
        #                            f'/home/sareeves/software/MutateEverything/data/{name}/msa/{code}{chain}/{code}{chain}.a3m')

KeyboardInterrupt: 

In [ ]:
import pandas as pd
import os
import shutil
from tqdm.notebook import tqdm

def apply_mutation(wt_seq, mutstring):
    mutlist = mutstring.split(':')
    mut_seq = list(wt_seq)
    for mut in mutlist:
        assert wt_seq[int(mut[1:-1])-1] == mut[0]
        mut_seq[int(mut[1:-1])-1] = mut[-1]

    return ''.join(mut_seq)

for df_path in ['/home/sareeves/PSLMs/data/preprocessed/GB1_Wu_2016_binding_domain.csv']:
    
    df = pd.read_csv(df_path, index_col=0)

    df['wt_seq'] = 'MTYKLILNGKTLKGETTTEAVDAATAEKVFKQYANDNGVDGEWTYDDATKTFTVTE'
    df['mut_seq'] = df.apply(
        lambda row: apply_mutation(row['wt_seq'], row['mut_type']), 
        axis=1
    )
    df = df.dropna(subset='wt_seq') 
    df['ddg'] = df['Fitness']

    df['pdb_id'] = df['code'] + df['chain']
    
    name = df_path.split('/')[-1].split('_')[0]
    os.makedirs(f'/home/sareeves/software/MutateEverything/data/dms/mutations/', exist_ok=True)
    df[['wt_seq', 'mut_seq', 'ddg', 'pdb_id', 'mut_info']].to_csv(f'/home/sareeves/software/MutateEverything/data/dms/mutations/GB1_Wu_2016_binding_domain.csv')

    df2 = pd.read_csv(f'/home/sareeves/software/MutateEverything/data/dms/mutations/GB1_Wu_2016_binding_domain.csv')
    df2 = parse_mutation_column_to_separate_columns(df2, 'mut_info')
    df2.to_csv(f'/home/sareeves/software/MutateEverything/data/dms/mutations/GB1_Wu_2016_binding_domain.csv')

    #for (code, chain), group in df.groupby(['code', 'chain']):
    #    os.makedirs(f'/home/sareeves/software/MutateEverything/data/dms/fasta/{code}{chain}', exist_ok=True)
    #    with open(f'/home/sareeves/software/MutateEverything/data/dms/fasta/{code}{chain}/{code}{chain}.fasta', 'w') as f:
    #        wt_seq = group['wt_seq'].head(1).item()
    #        f.write(f'>{code}{chain}\n{wt_seq}')
